# 🫁 Privacy-Preserving Cancer Detection — Data Pipeline + CNN Training
**Chiranthan's Component | Federated Learning Project**

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Your dataset is already in Google Drive at `MyDrive/Data/` with this structure:
   ```
   MyDrive/Data/
     train/   ← adenocarcinoma/, large.cell.carcinoma/, squamous.cell.carcinoma/, normal/
     valid/   ← same 4 class folders
     test/    ← same 4 class folders
   ```
3. Run all cells **top to bottom** (Runtime → Run all)

---

## Cell 1 — Verify GPU

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    for line in result.stdout.split('\n')[:12]:
        print(line)
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\nTensorFlow version: {tf.__version__}')
print(f'GPUs available:     {len(gpus)}')
for g in gpus:
    print(f'  {g}')

## Cell 2 — Mount Google Drive & verify dataset

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Edit these paths if your Drive layout is different ───────────────────────
DRIVE_DATA_DIR    = '/content/drive/MyDrive/Data'        # folder with train/ valid/ test/
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/FL_Project'  # where outputs will be saved

print('\nVerifying dataset structure on Drive...')
SPLITS   = ['train', 'valid', 'test']
CLASSES  = ['adenocarcinoma', 'large.cell.carcinoma', 'squamous.cell.carcinoma', 'normal']
all_ok   = True

for split in SPLITS:
    split_path = os.path.join(DRIVE_DATA_DIR, split)
    if not os.path.isdir(split_path):
        print(f'  ❌  {split}/ NOT FOUND at {split_path}')
        all_ok = False
        continue
    found_classes = sorted(os.listdir(split_path))
    total_imgs    = sum(
        len(os.listdir(os.path.join(split_path, c)))
        for c in found_classes if os.path.isdir(os.path.join(split_path, c))
    )
    print(f'  ✅  {split}/  →  classes: {found_classes}  ({total_imgs} images)')

if not all_ok:
    raise FileNotFoundError(
        f'One or more split folders missing under {DRIVE_DATA_DIR}.\n'
        f'Expected: train/, valid/, test/  — update DRIVE_DATA_DIR above.'
    )
print('\n✅  Dataset verified on Drive.')

## Cell 3 — Install extra dependencies

In [ ]:
# Colab already has TF, numpy, matplotlib — only install what's missing
!pip install -q opencv-python-headless scikit-learn tqdm
print('✅  Dependencies installed.')

## Cell 4 — Merge train/ + valid/ + test/ → data/raw/\<class\>/

We pool **all** images first so `preprocess.py` sees the full dataset,  
then `partition.py` creates its own reproducible stratified split (seed=42).

In [ ]:
import shutil, os

PROJECT_ROOT = '/content/FL_Project'
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Classes must match the CLASSES list in ml/preprocess.py exactly
CLASSES = ['adenocarcinoma', 'large.cell.carcinoma', 'squamous.cell.carcinoma', 'normal']
SPLITS  = ['train', 'valid', 'test']

print('Merging Drive splits → data/raw/<class>/  ...')
print('(train + valid + test are all pooled; partition.py will re-split)')
print()

grand_total = 0
for cls in CLASSES:
    dst_cls = os.path.join(RAW_DATA_DIR, cls)
    os.makedirs(dst_cls, exist_ok=True)
    cls_total = 0
    for split in SPLITS:
        src_cls = os.path.join(DRIVE_DATA_DIR, split, cls)
        if not os.path.isdir(src_cls):
            print(f'  [WARN] {split}/{cls} not found — skipping')
            continue
        files = [f for f in os.listdir(src_cls)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for fname in files:
            # Prefix split name to avoid filename collisions across splits
            dst_name = f'{split}_{fname}'
            shutil.copy2(os.path.join(src_cls, fname),
                         os.path.join(dst_cls, dst_name))
        cls_total += len(files)
    print(f'  ✅  {cls:<30}  →  {cls_total} images')
    grand_total += cls_total

print(f'\nTotal images copied into data/raw/: {grand_total}')
print('\n✅  Dataset ready for preprocessing.')

## Cell 5 — Create remaining project folder structure

In [ ]:
import os

dirs = [
    f'{PROJECT_ROOT}/ml',
    f'{PROJECT_ROOT}/data/node1',
    f'{PROJECT_ROOT}/data/node2',
    f'{PROJECT_ROOT}/data/node3',
    f'{PROJECT_ROOT}/data/test',
    f'{PROJECT_ROOT}/models',
    f'{PROJECT_ROOT}/results',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓ {d}')

print('\n✅  Folder structure ready.')

## Cell 6 — Copy source files from Drive → Colab VM

Upload `model.py`, `explore_data.py`, `preprocess.py`, `partition.py`, `train_centralized.py`  
to `MyDrive/FL_Project/` on Drive — then this cell copies them automatically.

In [ ]:
import shutil, os

ML_DIR   = f'{PROJECT_ROOT}/ml'
ML_FILES = [
    'model.py', 'explore_data.py', 'preprocess.py',
    'partition.py', 'train_centralized.py'
]

print('Copying source files from Drive...')
for fname in ML_FILES:
    src = os.path.join(DRIVE_PROJECT_DIR, fname)
    dst = os.path.join(ML_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  ✅  {fname}')
    else:
        print(f'  ❌  {fname} not found at {src}')
        print(f'       → Upload it to MyDrive/FL_Project/ and re-run this cell.')

# Write class_map.json
import json
class_map = {"0": "adenocarcinoma", "1": "large.cell.carcinoma",
             "2": "squamous.cell.carcinoma", "3": "normal"}
with open(f'{PROJECT_ROOT}/data/class_map.json', 'w') as f:
    json.dump(class_map, f, indent=2)
print('  ✅  class_map.json')

## Cell 7 — Step 1: Run `explore_data.py` (EDA)

In [ ]:
import subprocess, sys, os

os.chdir(f'{PROJECT_ROOT}/ml')

print('═'*60)
print('  Running explore_data.py ...')
print('═'*60)

result = subprocess.run([sys.executable, 'explore_data.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
if result.returncode != 0:
    raise RuntimeError('explore_data.py failed')

print('\n✅  explore_data.py finished.')

sample_grid = f'{PROJECT_ROOT}/data/sample_grid.png'
if os.path.exists(sample_grid):
    from IPython.display import Image, display
    display(Image(sample_grid))

## Cell 8 — Step 2: Run `preprocess.py` (Images → NumPy arrays)

In [ ]:
import subprocess, sys, os

print('═'*60)
print('  Running preprocess.py ...')
print('═'*60)

result = subprocess.run([sys.executable, 'preprocess.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
if result.returncode != 0:
    raise RuntimeError('preprocess.py failed')

for fname in ['X_all.npy', 'y_all.npy']:
    path = f'{PROJECT_ROOT}/data/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**2)
        print(f'  ✓ {fname}  ({size:.1f} MB)')
    else:
        print(f'  ✗ {fname} NOT found')

print('\n✅  preprocess.py finished.')

## Cell 9 — Step 3: Run `partition.py` (Arrays → Federated node splits)

In [ ]:
import subprocess, sys, os

print('═'*60)
print('  Running partition.py ...')
print('═'*60)

result = subprocess.run([sys.executable, 'partition.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
if result.returncode != 0:
    raise RuntimeError('partition.py failed')

import numpy as np
for node in ['node1', 'node2', 'node3', 'test']:
    xp = f'{PROJECT_ROOT}/data/{node}/X.npy'
    if os.path.exists(xp):
        X = np.load(xp)
        print(f'  ✓ {node}: X.npy shape = {X.shape}')
    else:
        print(f'  ✗ {node}/X.npy not found')

print('\n✅  partition.py finished.')

## Cell 10 — Step 4: Run `train_centralized.py` (Train CNN on GPU)

In [ ]:
import subprocess, sys, os

print('═'*60)
print('  Running train_centralized.py  [GPU-accelerated]...')
print('═'*60)

# Stream output live so you can watch epoch progress
process = subprocess.Popen(
    [sys.executable, 'train_centralized.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='')
process.wait()
if process.returncode != 0:
    raise RuntimeError('train_centralized.py failed')

for path in [f'{PROJECT_ROOT}/models/centralized_model.h5',
             f'{PROJECT_ROOT}/results/centralized_metrics.json']:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**2)
        print(f'  ✓ {os.path.basename(path)}  ({size:.2f} MB)')
    else:
        print(f'  ✗ {os.path.basename(path)} NOT found')

print('\n✅  Training complete!')

## Cell 11 — Display training metrics

In [ ]:
import json, os

metrics_path = f'{PROJECT_ROOT}/results/centralized_metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('Centralized Baseline Metrics')
    print('─'*35)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k:<20}: {v:.4f}')
        else:
            print(f'  {k:<20}: {v}')
else:
    print('metrics file not found — check train_centralized.py')

## Cell 12 — Save outputs back to Google Drive

In [ ]:
import shutil, os

DRIVE_OUT = DRIVE_PROJECT_DIR

FOLDERS_TO_SAVE = {
    f'{PROJECT_ROOT}/models':     f'{DRIVE_OUT}/models',
    f'{PROJECT_ROOT}/results':    f'{DRIVE_OUT}/results',
    f'{PROJECT_ROOT}/data/node1': f'{DRIVE_OUT}/data/node1',
    f'{PROJECT_ROOT}/data/node2': f'{DRIVE_OUT}/data/node2',
    f'{PROJECT_ROOT}/data/node3': f'{DRIVE_OUT}/data/node3',
    f'{PROJECT_ROOT}/data/test':  f'{DRIVE_OUT}/data/test',
}
FILES_TO_SAVE = {
    f'{PROJECT_ROOT}/data/X_all.npy':      f'{DRIVE_OUT}/data/X_all.npy',
    f'{PROJECT_ROOT}/data/y_all.npy':      f'{DRIVE_OUT}/data/y_all.npy',
    f'{PROJECT_ROOT}/data/class_map.json': f'{DRIVE_OUT}/data/class_map.json',
    f'{PROJECT_ROOT}/data/sample_grid.png':f'{DRIVE_OUT}/data/sample_grid.png',
}

os.makedirs(DRIVE_OUT, exist_ok=True)

print('Copying folders to Drive...')
for src, dst in FOLDERS_TO_SAVE.items():
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  ✓ {os.path.basename(src)}/ → Drive')
    else:
        print(f'  ✗ {src} not found — skipping')

print('\nCopying files to Drive...')
for src, dst in FILES_TO_SAVE.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / (1024**2)
        print(f'  ✓ {os.path.basename(src)}  ({size:.1f} MB) → Drive')
    else:
        print(f'  ✗ {src} not found — skipping')

print()
print('✅  All outputs saved to Google Drive.')
print(f'   Location: {DRIVE_OUT}/')
print()
print('Share with teammates:')
print('  • Rohith  → data/node1/, node2/, node3/ + ml/model.py')
print('  • Shree   → data/node1/, node2/, node3/ (reads X.npy, y.npy)')
print('  • Pavan   → models/centralized_model.h5 + data/class_map.json')

---
# 🌐 Part 2 — Federated Learning Training

Now that the centralized baseline is done, we run the same CNN across **3 simulated hospital nodes** using **Flower (FedAvg)**.

The data partitions (`data/node1/`, `node2/`, `node3/`, `test/`) were already created by `partition.py` in Cell 9.

---

## Cell 13 — Install Flower

In [ ]:
!pip install -q "flwr>=1.12" hatchling
print("✅  Flower installed.")

## Cell 14 — Copy FL source files from Drive → VM

In [ ]:
import shutil, os

FL_SRC  = os.path.join(DRIVE_PROJECT_DIR, "federated_learning")
FL_DST  = f"{PROJECT_ROOT}/federated_learning"

if os.path.exists(FL_SRC):
    if os.path.exists(FL_DST):
        shutil.rmtree(FL_DST)
    shutil.copytree(FL_SRC, FL_DST)
    print("✅  Copied federated_learning/ from Drive")
    print("    Files:", os.listdir(FL_DST))
else:
    print(f"❌  {FL_SRC} not found on Drive!")
    print("    Upload the federated_learning/ folder to MyDrive/FL_Project/")

## Cell 15 — Verify all required FL files exist

In [ ]:
import os

required_files = [
    "ml/model.py",
    "federated_learning/task.py",
    "federated_learning/client_app.py",
    "federated_learning/server_app.py",
    "federated_learning/pyproject.toml",
    "federated_learning/__init__.py",
    "data/node1/X.npy",
    "data/node1/y.npy",
    "data/node2/X.npy",
    "data/node2/y.npy",
    "data/node3/X.npy",
    "data/node3/y.npy",
    "data/test/X.npy",
    "data/test/y.npy",
]

all_ok = True
for f in required_files:
    path = os.path.join(PROJECT_ROOT, f)
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    if not exists:
        all_ok = False
    print(f"  {status}  {f}")

if not all_ok:
    raise FileNotFoundError(
        "Some files are missing! Upload them to MyDrive/FL_Project/ on Google Drive."
    )
else:
    print("
✅  All files present. Ready for Federated Learning!")

## Cell 16 — Run Federated Learning Training (10 rounds, FedAvg, 3 nodes)

In [ ]:
import subprocess, sys, os

os.chdir(f"{PROJECT_ROOT}/federated_learning")

print("═" * 60)
print("  FL Training Starting")
print("  Strategy:   FedAvg")
print("  Rounds:     10")
print("  Nodes:      3 (simulated)")
print("  Local epochs per round: 3")
print("═" * 60)
print()

process = subprocess.Popen(
    [sys.executable, "-m", "flwr", "run", ".", "--stream"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode == 0:
    print("
" + "═" * 60)
    print("  ✅  FL Training Complete!")
    print("═" * 60)
else:
    print(f"
❌  Training failed with return code {process.returncode}")
    print("    Check the output above for errors.")

## Cell 17 — Verify FL model and metrics were created

In [ ]:
import os

fl_outputs = {
    "federated_model.h5": f"{PROJECT_ROOT}/models/federated_model.h5",
    "fl_metrics.json":    f"{PROJECT_ROOT}/results/fl_metrics.json",
}

for name, path in fl_outputs.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✅  {name}  ({size:.2f} MB)")
    else:
        print(f"  ❌  {name}  NOT FOUND")
        print(f"       Expected at: {path}")

## Cell 18 — Display FL metrics (compare with centralized baseline)

In [ ]:
import json, os

print("═" * 50)
print("  Federated Learning Metrics")
print("═" * 50)

fl_metrics_path = f"{PROJECT_ROOT}/results/fl_metrics.json"
if os.path.exists(fl_metrics_path):
    with open(fl_metrics_path) as f:
        fl_metrics = json.load(f)
    for k, v in fl_metrics.items():
        if isinstance(v, float):
            print(f"  {k:<25}: {v:.4f}")
        elif isinstance(v, dict):
            print(f"  {k}:")
            for kk, vv in v.items():
                val = f"{vv:.4f}" if isinstance(vv, float) else str(vv)
                print(f"    {kk:<23}: {val}")
        else:
            print(f"  {k:<25}: {v}")
else:
    print("  ❌  fl_metrics.json not found")

# Compare with centralized baseline if available
cent_path = f"{PROJECT_ROOT}/results/centralized_metrics.json"
if os.path.exists(cent_path):
    with open(cent_path) as f:
        cent = json.load(f)
    print()
    print("─" * 50)
    print("  Centralized Baseline (for comparison)")
    print("─" * 50)
    for k, v in cent.items():
        if isinstance(v, float):
            print(f"  {k:<25}: {v:.4f}")
        else:
            print(f"  {k:<25}: {v}")

## Cell 19 — Save FL outputs back to Google Drive

In [ ]:
import shutil, os

items_to_save = {
    f"{PROJECT_ROOT}/models/federated_model.h5":   f"{DRIVE_PROJECT_DIR}/models/federated_model.h5",
    f"{PROJECT_ROOT}/results/fl_metrics.json":     f"{DRIVE_PROJECT_DIR}/results/fl_metrics.json",
}

print("Saving FL outputs to Drive...")
for src, dst in items_to_save.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / (1024 * 1024)
        print(f"  ✅  {os.path.basename(src)}  ({size:.2f} MB) → Drive")
    else:
        print(f"  ❌  {src} not found")

print()
print("✅  Done! federated_model.h5 is now on your Google Drive.")
print(f"   Location: {DRIVE_PROJECT_DIR}/models/")

## Cell 20 — Summary

✅ **Pipeline complete!** You now have:

| Output | Location on Drive |
|---|---|
| Centralized model | `FL_Project/models/centralized_model.h5` |
| Federated model | `FL_Project/models/federated_model.h5` |
| Centralized metrics | `FL_Project/results/centralized_metrics.json` |
| FL metrics | `FL_Project/results/fl_metrics.json` |
| Node data partitions | `FL_Project/data/node{1,2,3}/` |
| Test set | `FL_Project/data/test/` |

**Next steps:**
- Share `federated_model.h5` with Pavan for backend integration
- Compare centralized vs federated accuracy in the metrics files
- Run Shree's synthetic data augmentation on the same node partitions